# Manage Fabric connections at scale

> **Disclaimer**
>
> This notebook contains sample code provided for demonstration purposes only. It isn't an official Microsoft product or supported solution and is provided **as is**, without warranties of any kind. Use it only in a test or other nonproduction environment. You are responsible for reviewing, testing, securing, and validating the code before use and assume all risks arising from its use, including any changes to connections, permissions, credentials, or data.

This notebook uses the Microsoft Fabric Connections REST APIs to identify three governance risks:

- **Stale connections** that are not bound to an item or whose credentials haven't been used recently.
- **Duplicate connections** that point to the same endpoint through the same connectivity route.
- **Ownership risks** where the only owner is an individual user.

> This notebook is designed for the **Fabric Python runtime**, not the PySpark runtime. It uses Pandas and `requests`; no Spark session or lakehouse is required.

All executable cells are read-only. The final optional cell shows how to add an approved Microsoft Entra group as an owner, but every line is commented out so normal execution can't change ownership.

## Prerequisites and configuration

Run the notebook as an identity that can access the connections you want to review. Listing connections and role assignments requires `Connection.Read.All` or `Connection.ReadWrite.All`. Adding an owner requires `Connection.ReadWrite.All` and sufficient rights on each connection.

The API returns only connections visible to the caller. For cloud connections, this generally means connections the caller owns. A gateway administrator can see connections on gateways they administer.

The configuration cell applies these rules:

- Connections created before **May 1, 2026** are excluded from the unbound test because Connection Recency wasn't available when they were created.
- Credentials not used in the last **90 days**, including credentials with no recorded use, are flagged.
- `APPROVED_OWNER_GROUP_ID` holds the object ID of the Microsoft Entra security group that could be added as an owner. Leave it blank for read-only analysis.

In [ ]:
from datetime import datetime, timezone
import time
from urllib.parse import urlparse

import notebookutils
import pandas as pd
import requests

FABRIC_API_ROOT = "https://api.fabric.microsoft.com/v1"
RECENCY_START_DATE = pd.Timestamp("2026-05-01", tz="UTC")
CREDENTIAL_UNUSED_DAYS = 90
APPROVED_OWNER_GROUP_ID = ""
DATE_DISPLAY_FORMAT = "%m/%d/%Y"

analysis_time = pd.Timestamp(datetime.now(timezone.utc))
credential_usage_cutoff = analysis_time - pd.Timedelta(days=CREDENTIAL_UNUSED_DAYS)

def format_dates_for_display(dataframe):
    display_df = dataframe.copy()
    date_columns = [
        column for column in display_df.columns
        if column.endswith("DateTime")
    ]
    for column in date_columns:
        display_df[column] = display_df[column].dt.strftime(DATE_DISPLAY_FORMAT)
    return display_df

print(f"Analysis date (UTC): {analysis_time.strftime(DATE_DISPLAY_FORMAT)}")
print(f"Unbound recency start date: {RECENCY_START_DATE.strftime(DATE_DISPLAY_FORMAT)}")
print(f"Credential usage cutoff date (UTC): {credential_usage_cutoff.strftime(DATE_DISPLAY_FORMAT)}")


## Authenticate and load the connection inventory

Fabric provides an access token for the notebook's current identity through `notebookutils`. The request helper honors the service's `Retry-After` response when throttled. The pagination helper follows `continuationUri` until every accessible page is collected and accepts continuation URLs only from `api.fabric.microsoft.com`.

The Connections API nests endpoint, credential, and recency properties. The final helper flattens the fields needed for analysis into a Pandas DataFrame and parses all dates as UTC timestamps.

In [ ]:
access_token = notebookutils.credentials.getToken("pbi")
session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
})

def fabric_request(method, url, *, max_attempts=6, **kwargs):
    for attempt in range(1, max_attempts + 1):
        response = session.request(method, url, timeout=60, **kwargs)
        if response.status_code != 429:
            response.raise_for_status()
            return response
        if attempt == max_attempts:
            response.raise_for_status()
        retry_after = int(response.headers.get("Retry-After", "5"))
        print(f"Request throttled. Retrying in {retry_after} seconds...")
        time.sleep(retry_after)

def get_all_pages(url):
    records = []
    next_url = url
    while next_url:
        parsed_url = urlparse(next_url)
        if parsed_url.scheme != "https" or parsed_url.netloc != "api.fabric.microsoft.com":
            raise ValueError(f"Unexpected continuation URL: {next_url}")
        body = fabric_request("GET", next_url).json()
        records.extend(body.get("value", []))
        next_url = body.get("continuationUri")
    return records

def flatten_connection(connection):
    details = connection.get("connectionDetails") or {}
    credentials = connection.get("credentialDetails") or {}
    recency = connection.get("connectionRecency") or {}
    return {
        "ConnectionID": connection.get("id"),
        "DisplayName": connection.get("displayName"),
        "ConnectionType": details.get("type"),
        "ConnectionPath": details.get("path"),
        "ConnectivityType": connection.get("connectivityType"),
        "GatewayID": connection.get("gatewayId"),
        "PrivacyLevel": connection.get("privacyLevel"),
        "CredentialType": credentials.get("credentialType"),
        "SingleSignOnType": credentials.get("singleSignOnType"),
        "ConnectionEncryption": credentials.get("connectionEncryption"),
        "CreatedDateTime": recency.get("createdDateTime"),
        "LastBoundDateTime": recency.get("lastBoundDateTime"),
        "LastCredentialUsedDateTime": recency.get("lastCredentialUsedDateTime"),
    }

connection_records = get_all_pages(f"{FABRIC_API_ROOT}/connections")
connections_df = pd.DataFrame([flatten_connection(item) for item in connection_records])
for column in ["CreatedDateTime", "LastBoundDateTime", "LastCredentialUsedDateTime"]:
    connections_df[column] = pd.to_datetime(connections_df[column], errors="coerce", utc=True)

print(f"Loaded {len(connections_df):,} accessible connections.")
display(format_dates_for_display(connections_df))


# Section 1: Stale connections

A connection is flagged when either condition is true:

- **Unbound:** `LastBoundDateTime` is null and `CreatedDateTime` is on or after May 1, 2026. Earlier connections are excluded because a null value might reflect missing historical recency data.
- **Credentials unused:** `LastCredentialUsedDateTime` is null or earlier than the 90-day cutoff. A null value means no credential use is recorded.

These are review candidates, not automatic deletion candidates. Confirm seasonal, annual, and incident-response workloads with connection owners before making changes.

In [ ]:
created_after_recency_start = connections_df["CreatedDateTime"].ge(RECENCY_START_DATE)
is_unbound = connections_df["LastBoundDateTime"].isna() & created_after_recency_start
credentials_unused = (
    connections_df["LastCredentialUsedDateTime"].isna()
    | connections_df["LastCredentialUsedDateTime"].lt(credential_usage_cutoff)
)
stale_mask = is_unbound | credentials_unused
stale_connections_df = connections_df.loc[stale_mask].copy()
stale_connections_df["IsUnbound"] = is_unbound.loc[stale_mask]
stale_connections_df["CredentialsUnused90Days"] = credentials_unused.loc[stale_mask]

def describe_stale_reason(row):
    reasons = []
    if row["IsUnbound"]:
        reasons.append("No binding recorded")
    if row["CredentialsUnused90Days"]:
        reason = (
            "No credential use recorded"
            if pd.isna(row["LastCredentialUsedDateTime"])
            else f"Credentials unused for more than {CREDENTIAL_UNUSED_DAYS} days"
        )
        reasons.append(reason)
    return "; ".join(reasons)

stale_connections_df["ReviewReason"] = stale_connections_df.apply(describe_stale_reason, axis=1)
stale_connections_df = stale_connections_df.sort_values(
    ["IsUnbound", "LastCredentialUsedDateTime"],
    ascending=[False, True],
    na_position="first",
)
stale_output_columns = [
    "ConnectionID", "DisplayName", "ConnectionType", "ConnectionPath",
    "ConnectivityType", "GatewayID", "CreatedDateTime",
    "LastBoundDateTime", "LastCredentialUsedDateTime", "IsUnbound",
    "CredentialsUnused90Days", "ReviewReason",
]
print(f"Flagged {len(stale_connections_df):,} stale connection candidates.")
display(format_dates_for_display(stale_connections_df[stale_output_columns]))


# Section 2: Duplicate connections

Connections are duplicates when `ConnectionType`, `ConnectionPath`, `ConnectivityType`, and `GatewayID` all match. No additional field is universally required.

You can make the comparison stricter by adding `CredentialType`, `SingleSignOnType`, `ConnectionEncryption`, or `PrivacyLevel`. They're shown for review but excluded from the key because connections to the same endpoint can still be consolidation candidates when authentication or privacy settings differ.

Each duplicate set receives a stable review label such as `DUP-001`. Rows are sorted by that label so every group appears together. Within each group, the output identifies the connection with the most recent credential use. A tie or a group with no usage history remains a manual review decision.

In [ ]:
DUPLICATE_KEY_COLUMNS = [
    "ConnectionType", "ConnectionPath", "ConnectivityType", "GatewayID"
]
duplicate_size = connections_df.groupby(
    DUPLICATE_KEY_COLUMNS, dropna=False
)["ConnectionID"].transform("size")
duplicates_df = connections_df.loc[duplicate_size.gt(1)].copy()
duplicates_df["DuplicateCount"] = duplicate_size.loc[duplicate_size.gt(1)]

if not duplicates_df.empty:
    duplicate_group_number = duplicates_df.groupby(
        DUPLICATE_KEY_COLUMNS, dropna=False, sort=True
    ).ngroup() + 1
    group_width = max(3, len(str(duplicate_group_number.max())))
    duplicates_df["DuplicateGroup"] = duplicate_group_number.map(
        lambda number: f"DUP-{number:0{group_width}d}"
    )
    duplicates_df["MostRecentUseInGroup"] = duplicates_df.groupby(
        DUPLICATE_KEY_COLUMNS, dropna=False
    )["LastCredentialUsedDateTime"].transform("max")
    duplicates_df["MostRecentlyUsedCandidate"] = (
        duplicates_df["LastCredentialUsedDateTime"].notna()
        & duplicates_df["LastCredentialUsedDateTime"].eq(duplicates_df["MostRecentUseInGroup"])
    )
    duplicates_df["MostRecentCandidateCount"] = duplicates_df.groupby(
        DUPLICATE_KEY_COLUMNS, dropna=False
    )["MostRecentlyUsedCandidate"].transform("sum")

    def duplicate_guidance(row):
        if pd.isna(row["MostRecentUseInGroup"]):
            return "No usage history; review all connections"
        if row["MostRecentlyUsedCandidate"] and row["MostRecentCandidateCount"] == 1:
            return "Most recently used; preferred keep candidate"
        if row["MostRecentlyUsedCandidate"]:
            return "Tied for most recent use; review manually"
        return "Older usage; review for consolidation"

    duplicates_df["ReviewGuidance"] = duplicates_df.apply(duplicate_guidance, axis=1)
    duplicates_df = duplicates_df.sort_values(
        ["DuplicateGroup", "LastCredentialUsedDateTime", "DisplayName"],
        ascending=[True, False, True],
        na_position="last",
    )
else:
    duplicates_df["DuplicateGroup"] = pd.Series(dtype="string")

duplicate_output_columns = [
    "DuplicateGroup", "ConnectionID", "DisplayName", *DUPLICATE_KEY_COLUMNS,
    "CredentialType",
    "SingleSignOnType", "ConnectionEncryption", "PrivacyLevel",
    "LastCredentialUsedDateTime", "DuplicateCount", "ReviewGuidance",
]
duplicate_group_count = (
    duplicates_df.groupby(DUPLICATE_KEY_COLUMNS, dropna=False).ngroups
    if not duplicates_df.empty else 0
)
print(f"Found {duplicate_group_count:,} groups containing {len(duplicates_df):,} connections.")
display(format_dates_for_display(duplicates_df[duplicate_output_columns]))


# Section 3: Ownership continuity

This section retrieves all paginated role assignments for every accessible connection and summarizes **all owners** in one row per connection. The owner list includes every principal with the `Owner` role, not only connections with a single owner.

For each owner, the output uses `displayName` from the Fabric `UserPrincipal` object and `userPrincipalName` from its `UserDetails` object. These values remain null when the API doesn't return them; the notebook doesn't resolve or substitute identity names from another service. Principal IDs are shown separately.

Every inventoried connection remains in the ownership summary. The separate risk table flags only connections whose role assignments were read successfully and whose complete owner set contains exactly one assignment with principal type **User**. Failed role-assignment requests are marked as unavailable and aren't classified.

In [ ]:
role_assignment_rows = []
role_assignment_errors = []
role_assignments_loaded_for = []

for connection in connections_df.itertuples(index=False):
    url = f"{FABRIC_API_ROOT}/connections/{connection.ConnectionID}/roleAssignments"
    try:
        for assignment in get_all_pages(url):
            principal = assignment.get("principal") or {}
            user_details = principal.get("userDetails") or {}
            group_details = principal.get("groupDetails") or {}
            role_assignment_rows.append({
                "ConnectionID": connection.ConnectionID,
                "DisplayName": connection.DisplayName,
                "RoleAssignmentID": assignment.get("id"),
                "Role": assignment.get("role"),
                "PrincipalID": principal.get("id"),
                "PrincipalType": principal.get("type"),
                "PrincipalDisplayName": principal.get("displayName"),
                "UserPrincipalName": user_details.get("userPrincipalName"),
                "GroupType": group_details.get("groupType"),
            })
        role_assignments_loaded_for.append(connection.ConnectionID)
    except requests.HTTPError as error:
        response = error.response
        role_assignment_errors.append({
            "ConnectionID": connection.ConnectionID,
            "DisplayName": connection.DisplayName,
            "StatusCode": response.status_code if response is not None else None,
            "Error": str(error),
        })

role_assignments_df = pd.DataFrame(role_assignment_rows, columns=[
    "ConnectionID", "DisplayName", "RoleAssignmentID", "Role",
    "PrincipalID", "PrincipalType", "PrincipalDisplayName",
    "UserPrincipalName", "GroupType",
])
role_assignment_errors_df = pd.DataFrame(role_assignment_errors)
print(f"Loaded {len(role_assignments_df):,} role assignments.")
if not role_assignment_errors_df.empty:
    print(f"Could not read assignments for {len(role_assignment_errors_df):,} connections.")
    display(role_assignment_errors_df)

owners_df = role_assignments_df.loc[role_assignments_df["Role"].eq("Owner")].copy()
owners_by_connection = {
    connection_id: owners.copy()
    for connection_id, owners in owners_df.groupby("ConnectionID", sort=False)
}
assignment_error_ids = set(role_assignment_errors_df.get("ConnectionID", pd.Series(dtype="string")))
ownership_rows = []
for connection in connections_df.itertuples(index=False):
    connection_owners = owners_by_connection.get(connection.ConnectionID, owners_df.iloc[0:0])
    assignments_available = connection.ConnectionID not in assignment_error_ids
    owner_count = len(connection_owners) if assignments_available else pd.NA
    owner_display_names = sorted(
        set(connection_owners["PrincipalDisplayName"].dropna().astype(str))
    )
    owner_user_principal_names = sorted(
        set(connection_owners["UserPrincipalName"].dropna().astype(str))
    )
    owner_types = sorted(set(connection_owners["PrincipalType"].dropna().astype(str)))
    owner_ids = sorted(set(connection_owners["PrincipalID"].dropna().astype(str)))
    is_single_user_risk = (
        assignments_available
        and owner_count == 1
        and str(connection_owners.iloc[0]["PrincipalType"]).casefold() == "user"
    )
    ownership_rows.append({
        "ConnectionID": connection.ConnectionID,
        "DisplayName": connection.DisplayName,
        "ConnectionType": connection.ConnectionType,
        "ConnectionPath": connection.ConnectionPath,
        "ConnectivityType": connection.ConnectivityType,
        "GatewayID": connection.GatewayID,
        "OwnerCount": owner_count,
        "OwnerDisplayNames": "; ".join(owner_display_names) if owner_display_names else pd.NA,
        "OwnerUserPrincipalNames": (
            "; ".join(owner_user_principal_names)
            if owner_user_principal_names else pd.NA
        ),
        "OwnerTypes": "; ".join(owner_types) if owner_types else pd.NA,
        "OwnerPrincipalIDs": "; ".join(owner_ids) if owner_ids else pd.NA,
        "RoleAssignmentsStatus": "Available" if assignments_available else "Unavailable",
        "IsSingleUserOwnerRisk": is_single_user_risk,
        "ReviewReason": (
            "Only Owner role assignment is a User"
            if is_single_user_risk
            else (
                "Role assignments unavailable; not classified"
                if not assignments_available
                else "No single-user ownership risk identified"
            )
        ),
    })

ownership_summary_df = pd.DataFrame(ownership_rows, columns=[
    "ConnectionID", "DisplayName", "ConnectionType", "ConnectionPath",
    "ConnectivityType", "GatewayID", "OwnerCount", "OwnerDisplayNames",
    "OwnerUserPrincipalNames", "OwnerTypes", "OwnerPrincipalIDs",
    "RoleAssignmentsStatus", "IsSingleUserOwnerRisk", "ReviewReason",
]).sort_values(["IsSingleUserOwnerRisk", "DisplayName"], ascending=[False, True])
ownership_risks_df = ownership_summary_df.loc[
    ownership_summary_df["IsSingleUserOwnerRisk"]
].copy()
print(
    f"Reviewed {len(ownership_summary_df):,} connections and flagged "
    f"{len(ownership_risks_df):,} single-user ownership risks."
)
display(ownership_summary_df)


## Preview the approved group assignment

Set `APPROVED_OWNER_GROUP_ID` in the configuration cell to an approved Microsoft Entra security group's object ID. The preview below shows which connections would receive the group as an additional owner; it sends no POST request.

Adding a group owner preserves the individual owner and doesn't change stored credentials. For OAuth connections tied to an individual account, separately review whether the credential should use an organizational identity.

In [ ]:
planned_owner_additions_df = ownership_risks_df[[
    "ConnectionID", "DisplayName", "OwnerDisplayNames",
    "OwnerUserPrincipalNames"
]].copy()
planned_owner_additions_df["ApprovedOwnerGroupID"] = (
    APPROVED_OWNER_GROUP_ID or "Not configured"
)
planned_owner_additions_df["PlannedRole"] = "Owner"
planned_owner_additions_df["Status"] = (
    "Ready for optional add-owner cell"
    if APPROVED_OWNER_GROUP_ID
    else "Set APPROVED_OWNER_GROUP_ID before enabling changes"
)
display(planned_owner_additions_df)


## Optional: add the approved group as an owner

The code below is intentionally disabled. Review the ownership results, confirm the Entra group object ID, and verify `Connection.ReadWrite.All` before enabling it. Uncomment it only after completing your organization's approval process. Each request adds the group as an `Owner`; it doesn't remove existing assignments.

In [ ]:
# if not APPROVED_OWNER_GROUP_ID:
#     raise ValueError("Set APPROVED_OWNER_GROUP_ID in the configuration cell.")
#
# for connection in ownership_risks_df.itertuples(index=False):
#     add_owner_url = (
#         f"{FABRIC_API_ROOT}/connections/{connection.ConnectionID}/roleAssignments"
#     )
#     payload = {
#         "principal": {"id": APPROVED_OWNER_GROUP_ID, "type": "Group"},
#         "role": "Owner",
#     }
#     response = fabric_request("POST", add_owner_url, json=payload)
#     print(
#         f"Added group owner to {connection.DisplayName} "
#         f"({connection.ConnectionID}): HTTP {response.status_code}"
#     )


## Review and next steps

Treat every result as evidence for review rather than an instruction to delete or change a connection. Confirm stale candidates with workload owners, inspect settings and dependencies before consolidating duplicates, and add a maintained Entra security group only after approval.

Official API references:

- [List Connections](https://learn.microsoft.com/rest/api/fabric/core/connections/list-connections)
- [List Connection Role Assignments](https://learn.microsoft.com/rest/api/fabric/core/connections/list-connection-role-assignments)
- [Add Connection Role Assignment](https://learn.microsoft.com/rest/api/fabric/core/connections/add-connection-role-assignment)